# 01 — COLMAP Initialization and 3D Gaussian Splatting

This notebook walks through the full pipeline from a set of monocular images to a trained 3D Gaussian Splatting (3DGS) model. The two stages map onto a clean conceptual division:

1. **Structure-from-Motion (SfM) with COLMAP** — a classical geometry pipeline that figures out *where* each camera was and what the scene looks like as a sparse cloud of 3D points.
2. **3D Gaussian Splatting** — a neural rendering method that takes those sparse points as a starting guess and optimizes a set of 3D Gaussians until they reproduce the original photographs when rendered.

Understanding how these two stages connect is the core lesson here. COLMAP does not produce a finished 3D model; it produces *initialization data* — camera poses and a coarse point cloud — that 3DGS then refines. If the initialization is weak, the Gaussian optimization starts in a poor part of parameter space, and render quality suffers before any neural learning even begins.

**Why endoscopy?**
The dataset comes from the [SCARED benchmark](https://arxiv.org/abs/2101.01133), which contains stereo video from a laparoscopic camera inside an abdominal cavity. Endoscopy is a demanding setting for SfM: tissue surfaces are smooth and textureless, lighting moves with the camera, and there is no sky, no rigid background, and no wide-baseline parallax to anchor scale. Failures that would only appear at the edges in outdoor scenes become central here.

**Learning goals:**
- Treat COLMAP as a set of inspectable intermediate data products, not a black box.
- Develop intuition for how *frame overlap* (the fraction of the scene shared between consecutive images) determines reconstruction quality.
- Connect sparse SfM geometry to the later quality of neural rendering.
- Practice **prediction before execution**: commit to a hypothesis, run the experiment, then reconcile.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/FelTris/camp2_3drecon.git"
REPO_NAME = "camp2_3drecon"


def running_in_colab():
    try:
        import google.colab  # noqa: F401
    except ModuleNotFoundError:
        return False
    return True


if running_in_colab():
    repo = Path("/content") / REPO_NAME
    if not (repo / ".git").exists():
        if repo.exists():
            raise FileExistsError(f"{repo} exists but is not a Git checkout. Remove it or choose another path.")
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    os.chdir(repo)
else:
    repo = Path.cwd()
    if repo.name == "notebooks":
        repo = repo.parent

REPO = repo.resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

mpl_config = REPO / "outputs" / ".mplconfig"
mpl_config.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_config))

print("Repository:", REPO)

## Install Dependencies

In Colab, run this cell before importing the reconstruction libraries. It installs the Python requirements and clones the external model repositories used by the notebooks. Local users can leave `INSTALL_DEPENDENCIES = False` if the environment is already set up.


In [ ]:
INSTALL_DEPENDENCIES = running_in_colab()

if INSTALL_DEPENDENCIES:
    subprocess.run(["bash", "scripts/install_reconstruction_stack.sh"], cwd=REPO, check=True)
else:
    print("Skipping dependency install outside Colab.")
    print("Set INSTALL_DEPENDENCIES = True and rerun this cell if your local environment is not set up.")

In [ ]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import pycolmap
import torch
from torch.utils.data import DataLoader

from camp3d.data import ScaredSubset
from camp3d.metrics import image_metrics
from camp3d.splats import (
    GaussianSplatModel,
    PosedImage,
    PosedImageDataset,
    interpolate_camera_path,
    make_gaussian_optimizers,
    render_posed_image,
    render_virtual_camera,
    train_one_epoch,
)
from camp3d.viz import read_rgb, save_render_gif, show_colmap_reconstruction_plotly, show_render_comparison, show_stereo_pair
from IPython.display import Image as DisplayImage

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

dataset = ScaredSubset(REPO / "scared_640")
print(dataset.validate())

## Start With The Images

Before running any pipeline, it pays to look at the raw data. Two things are immediately apparent in a stereo endoscopy frame:

**Texture.** Biological tissue has a characteristic matte appearance with subtle specular highlights from the light source mounted on the camera. The surface is largely smooth and lacks the strong, repeatable corners and edges that classical feature detectors (SIFT, SURF, ORB) rely on. This means COLMAP will find fewer features per image than it would on a textured outdoor scene, and those features will be noisier.

**The stereo pair.** This notebook uses only the *left* stream for reconstruction — a deliberate simplification to isolate the effect of frame overlap before adding the complexity of stereo constraints. Notebook 03 revisits the scene with both streams, which provides a known, calibrated baseline between cameras and makes scale recovery exact. For now, the right image is shown purely as a reminder that the dataset is richer than what we exploit here.


In [ ]:
left_path, right_path = dataset.stereo_pairs()[0]
show_stereo_pair(read_rgb(left_path), read_rgb(right_path))

## Prediction: What Will Frame Skipping Do?

This is a forced-prediction exercise. Write down your answers before running the next cell. The goal is not to be right — it is to surface the mental model you are currently using so you can update it against the actual measurements.

**Background: what does frame overlap do for COLMAP?**

COLMAP matches features across image pairs and then *triangulates* 3D points from matched pairs that see the same scene point from different angles. For triangulation to work, two conditions must hold:

1. **Matching**: the same physical point must appear in both images and be described by a similar feature vector.
2. **Baseline**: the two cameras must be far enough apart that the angular difference (parallax) is measurable, but close enough that the appearance has not changed too much.

When you take every 5th or every 10th frame from a video, you increase the baseline between consecutive pairs — which helps triangulation depth precision — but you also reduce overlap, which reduces the probability of a successful feature match and risks breaking the connectivity of the reconstruction graph. In the extreme case, COLMAP's incremental mapper cannot register a new image because it shares too few points with the already-reconstructed model.

**Specific failure modes to anticipate:**

- *Feature matching fails first* when the scene has changed so much between frames that descriptors do not match, even though the geometric baseline would have been fine.
- *Camera registration fails* when enough matches exist to compute a two-view geometry, but not enough inliers pass the RANSAC verification for PnP (Perspective-n-Point) registration.
- *Sparse point density drops* when cameras register successfully but the wide baseline means fewer image pairs share a given point, so fewer points survive triangulation.

Write down which of these you expect to dominate at 5× and 10× skipping, and why.


In [ ]:
image_paths = dataset.left_images()
colmap_use_gpu = torch.cuda.is_available()
colmap_gpu_index = "-1"  # "-1" lets COLMAP choose automatically; use "0" to force GPU 0.
experiments = {
    "all_frames": image_paths,
    "every_5th": image_paths[::5],
    "every_10th": image_paths[::10],
}

print("COLMAP GPU enabled:", colmap_use_gpu)
for name, paths in experiments.items():
    print(f"{name:>10}: {len(paths)} images")

## COLMAP Experiment Helpers

The helper functions defined in this cell manage the bookkeeping around running three separate COLMAP reconstructions. Understanding what they do — and why — builds intuition about how SfM pipelines are structured in practice.

**The COLMAP pipeline has three distinct stages:**

1. **Feature extraction** (`pycolmap.extract_features`) — runs SIFT on every image independently and stores keypoints and descriptors in a SQLite database. This step does not know about other images; it only sees one image at a time.

2. **Exhaustive feature matching** (`pycolmap.match_exhaustive`) — compares every image pair, finds putative descriptor matches, and runs a geometric verification step (estimating a fundamental matrix via RANSAC) to filter outliers. "Exhaustive" means all O(N²) pairs — practical for small sets, but sequential matching or vocabulary-tree matching is used for large datasets.

3. **Incremental mapping** (`pycolmap.incremental_mapping`) — selects a good initial pair, triangulates a seed model, then alternates between registering new cameras (PnP + RANSAC) and triangulating new 3D points until no more cameras can be added. Bundle adjustment runs periodically to jointly refine cameras and points.

**Caching.** The notebook checks whether a reconstruction already exists before running COLMAP. SfM is the most expensive part of the pipeline, so skipping it on reruns saves significant time. The image manifest records the exact set of filenames used; if it changes, the cached result is deleted and COLMAP reruns from scratch.

**`choose_reconstruction`.** COLMAP's incremental mapper can produce multiple disconnected *components* — sub-models that could not be merged into a single consistent reconstruction. This helper picks the component that registered the most cameras, which is the most informative one for our purposes.

**Coordinate frames.** The `image_camtoworld` and `camera_K` helpers extract two things every renderer needs:
- The **camera-to-world transform** (a 4×4 rigid-body matrix) — where the camera was in world space and which way it was pointing.
- The **intrinsic matrix K** — the mapping from 3D camera-space points to 2D pixel coordinates, encoding focal length and principal point.

COLMAP stores poses as *camera-from-world* (the inverse convention), so `image_camtoworld` inverts the stored matrix.


In [ ]:
workspace = REPO / "outputs" / "01_colmap"


def prepare_image_dir(name, paths):
    experiment_dir = workspace / name
    image_dir = experiment_dir / "images"
    manifest_path = experiment_dir / "image_manifest.txt"
    image_names = [path.name for path in paths]
    if experiment_dir.exists() and (
        not manifest_path.exists() or manifest_path.read_text().splitlines() != image_names
    ):
        shutil.rmtree(experiment_dir)
    image_dir.mkdir(parents=True, exist_ok=True)
    for src in paths:
        dst = image_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
    manifest_path.write_text("\n".join(image_names) + "\n")
    return experiment_dir, image_dir


def choose_reconstruction(reconstructions):
    if isinstance(reconstructions, dict):
        candidates = list(reconstructions.values())
    else:
        candidates = list(reconstructions)
    if not candidates:
        return None
    return max(candidates, key=lambda reconstruction: sum(image.has_pose for image in reconstruction.images.values()))


def run_colmap_experiment(name, paths):
    experiment_dir, image_dir = prepare_image_dir(name, paths)
    database_path = experiment_dir / "database.db"
    sparse_dir = experiment_dir / "sparse"
    sparse_dir.mkdir(parents=True, exist_ok=True)

    model_dirs = sorted(path for path in sparse_dir.iterdir() if path.is_dir())
    if model_dirs:
        reconstruction = choose_reconstruction(pycolmap.Reconstruction(path) for path in model_dirs)
    else:
        reader_options = pycolmap.ImageReaderOptions(camera_model="OPENCV")
        extraction_options = pycolmap.FeatureExtractionOptions(
            use_gpu=colmap_use_gpu,
            gpu_index=colmap_gpu_index,
        )
        matching_options = pycolmap.FeatureMatchingOptions(
            use_gpu=colmap_use_gpu,
            gpu_index=colmap_gpu_index,
        )
        pycolmap.extract_features(
            database_path,
            image_dir,
            reader_options=reader_options,
            extraction_options=extraction_options,
        )
        pycolmap.match_exhaustive(
            database_path,
            matching_options=matching_options,
        )
        reconstruction = choose_reconstruction(
            pycolmap.incremental_mapping(database_path, image_dir, sparse_dir)
        )
        if reconstruction is None:
            return {"name": name, "image_dir": image_dir, "reconstruction": None}
    return {"name": name, "image_dir": image_dir, "reconstruction": reconstruction}


def image_camtoworld(image):
    cam_from_world_attr = getattr(image, "cam_from_world")
    cam_from_world = cam_from_world_attr() if callable(cam_from_world_attr) else cam_from_world_attr
    world_to_cam = np.eye(4, dtype=np.float32)
    world_to_cam[:3, :4] = np.asarray(cam_from_world.matrix(), dtype=np.float32)
    return np.linalg.inv(world_to_cam)


def camera_K(camera):
    return np.asarray(camera.calibration_matrix(), dtype=np.float32)


def colmap_training_data(reconstruction, image_dir):
    frames = []
    if reconstruction is None:
        return frames, np.empty((0, 3), dtype=np.float32), np.empty((0, 3), dtype=np.float32)
    for image in reconstruction.images.values():
        if not image.has_pose:
            continue
        camera = reconstruction.cameras[image.camera_id]
        frames.append(
            PosedImage(
                image_path=image_dir / image.name,
                camtoworld=image_camtoworld(image),
                K=camera_K(camera),
            )
        )

    points = []
    colors = []
    for point in reconstruction.points3D.values():
        points.append(point.xyz)
        colors.append(np.asarray(point.color) / 255.0)
    return frames, np.asarray(points, dtype=np.float32), np.asarray(colors, dtype=np.float32)


def reconstruction_summary(name, requested, reconstruction, frames, points):
    if reconstruction is None:
        return {
            "experiment": name,
            "input_images": requested,
            "registered_images": 0,
            "sparse_points": 0,
            "mean_reprojection_error": np.nan,
        }
    errors = [float(point.error) for point in reconstruction.points3D.values() if np.isfinite(point.error)]
    return {
        "experiment": name,
        "input_images": requested,
        "registered_images": len(frames),
        "sparse_points": len(points),
        "mean_reprojection_error": float(np.mean(errors)) if errors else np.nan,
    }

In [ ]:
results = {}
summary_rows = []

for name, paths in experiments.items():
    print(f"\nRunning or loading: {name}")
    result = run_colmap_experiment(name, paths)
    frames, points, colors = colmap_training_data(result["reconstruction"], result["image_dir"])
    result.update({"frames": frames, "points": points, "colors": colors})
    results[name] = result
    summary_rows.append(reconstruction_summary(name, len(paths), result["reconstruction"], frames, points))

summary = pd.DataFrame(summary_rows)
display(summary)

## Inspect The Reconstructions

The sparse COLMAP output is not a mesh or a dense depth map — it is a set of **3D point tracks**: each point has a 3D position, a colour (averaged from the images that observed it), and a reprojection error (how far the projected point deviates from the observed 2D keypoints, in pixels).

**What the point cloud tells you:**

- *Dense regions* correspond to parts of the scene that were visible from many overlapping views and had enough texture for SIFT to find repeatable keypoints. In endoscopy this is often a specular highlight, a surgical tool edge, or a vascular marking.
- *Empty regions* do not mean the scene is empty there. They mean COLMAP could not reliably track features across that part of the scene — perhaps because the surface is textureless, or because the lighting changed between frames.
- *Camera frustums* show the registered poses. A gap in the frustum sequence means those frames were not registered — they either had too few matches to initialize, or the incremental mapper could not add them to the growing model.

**Reprojection error** is the per-point average distance (in pixels) between where COLMAP says the point should project and where the matched keypoint actually was. Values below ~1 pixel are typical for a well-calibrated system. Large errors indicate either a poor calibration model, a dynamic object that moved between frames, or a degenerate triangulation angle.

**The normalization note.** The visualization rescales everything into a unit cube so different experiments appear at a comparable scale in the viewer. The actual 3DGS training always uses the original COLMAP metric coordinates; this is purely a display convenience.


In [ ]:
def show_colmap_experiment(name):
    result = results[name]
    points = result["points"]
    frames = result["frames"]
    if len(points) == 0 or len(frames) == 0:
        print(f"{name}: no reconstruction to visualize")
        return None
    camtoworlds = np.stack([frame.camtoworld for frame in frames], axis=0)
    image_names = [frame.image_path.name for frame in frames]
    return show_colmap_reconstruction_plotly(
        points,
        result["colors"],
        camtoworlds=camtoworlds,
        image_names=image_names,
        max_points=50_000,
        point_size=2.0,
        title=f"{name}: sparse COLMAP points and cameras",
        normalize=True,
    )

### All Frames

With the full frame sequence, COLMAP has maximum overlap between consecutive images — typically 80–90% of the field of view is shared between adjacent frames. This gives the matcher plenty of opportunities to find correspondences, and the incremental mapper can usually register nearly every input image.

Expect a dense, well-connected point cloud that covers the visible surface. The camera frustums should form a smooth trajectory with no visible gaps. This is the *reference* reconstruction: any degradation you see in the 5× and 10× experiments is attributable to reduced overlap, not to the scene itself.


In [ ]:
fig = show_colmap_experiment("all_frames")
if fig is not None:
    fig.show()

### Every 5th Frame

At 5× skipping, consecutive frames in the experiment share roughly 50–70% of the field of view, depending on the speed of camera motion. This is still within the comfortable operating range of SIFT matching for most scenes, but endoscopy is not most scenes.

Look for:
- Whether COLMAP still registers most of the input images, or whether some frames are dropped.
- Whether the point cloud has visible holes compared to the all-frames case.
- Whether the mean reprojection error changes — wider baselines can actually *improve* triangulation accuracy for points that do register, because the parallax angle is larger.

The key question is whether the reconstruction remains a single connected component or breaks into fragments.


In [ ]:
fig = show_colmap_experiment("every_5th")
if fig is not None:
    fig.show()

### Every 10th Frame

At 10× skipping, consecutive input frames may share less than 30% of the field of view. This is near or past the limit for reliable exhaustive feature matching in a textureless scene.

Two outcomes are possible depending on the specific sequence:
1. **Graceful degradation** — COLMAP still finds enough matches to maintain connectivity, but registers fewer cameras and triangulates fewer points. The point cloud is sparse and the reprojection errors may be larger.
2. **Fragmentation** — the reconstruction graph breaks into disconnected sub-models. `choose_reconstruction` will pick the largest fragment, which may cover only a portion of the scene. Some experiments may return `None` if no sub-model reaches a minimum size.

Compare these plots quantitatively against the summary table from the previous cell: `registered_images` and `sparse_points` are the clearest signals.


In [ ]:
fig = show_colmap_experiment("every_10th")
if fig is not None:
    fig.show()

## Train 3DGS From The Full COLMAP Initialization

We now hand the COLMAP output — camera poses, intrinsics, and a sparse point cloud — to the 3D Gaussian Splatting optimizer. It is worth pausing to understand exactly what happens at this boundary.

**What 3DGS receives from COLMAP:**

- A set of **posed images**: each training image paired with the camera matrix K and the camera-to-world transform. These are the targets the model will try to reproduce.
- A **point cloud with colors**: each COLMAP point becomes the centre of one initial 3D Gaussian. The color of the point sets the initial colour of the Gaussian. The `init_scale` parameter controls the initial size of the Gaussians (here 0.01 scene units — small relative to the scene extent), and `init_opacity` (0.15) starts them mostly transparent.

**Why does initialization matter?**

3DGS is a differentiable renderer: it rasterizes the Gaussians into an image, computes a pixel-wise L1 loss against the target photograph, and backpropagates gradients through the rasterization to update the Gaussian parameters (mean, covariance, color, opacity). Like any gradient-based optimizer, it can get stuck in local minima if the starting configuration is too far from the ground truth.

A dense, accurate COLMAP point cloud seeds Gaussians near the true scene surface, so early renders already resemble the targets and gradients are informative. A sparse or inaccurate initialization seeds many Gaussians in empty space or the wrong location, and the optimizer must first move them to useful positions before it can start refining appearance.

**Image scaling.** The training images are loaded at 25% of their native resolution (`image_scale=0.25`). This is a common training strategy: lower-resolution images make each epoch faster and make the optimizer converge to coarse structure first. Full-resolution fine-tuning would follow in a production pipeline.


In [ ]:
baseline = results["all_frames"]
frames = baseline["frames"]
points = baseline["points"]
colors = baseline["colors"]

trainset = PosedImageDataset(frames, image_scale=0.25)
trainloader = DataLoader(trainset, batch_size=1, shuffle=True)

model = GaussianSplatModel(points, colors, init_scale=0.01, init_opacity=0.15).to(device)
optimizer = make_gaussian_optimizers(model, lr=1e-3)

print("train images:", len(trainset))
print("gaussians:", model.module.means.shape[0])

In [ ]:
for epoch in range(30):
    loss = train_one_epoch(model, trainloader, optimizer, device=device)
    print(f"epoch {epoch:02d} | L1 loss {loss:.4f}")

## Rendered Views And Image Metrics

After training, we render the scene from the original training camera poses and compare the renders to the ground-truth photographs. This is a **training-set evaluation** — we are measuring how well the model memorized the views it was trained on, not how well it generalizes to new viewpoints. Novel-view synthesis is tested in the next cell.

**PSNR (Peak Signal-to-Noise Ratio)**
PSNR is derived from per-pixel mean squared error:

$$\text{PSNR} = 10 \cdot \log_{10}\left(\frac{1}{\text{MSE}}\right)$$

Higher is better. A rough intuition: below ~20 dB the image looks noticeably blurry or wrong; above ~30 dB differences are subtle; above ~40 dB is near-lossless for natural images. Because it is based on MSE, it penalizes any pixel that is off by a lot — including small spatial misalignments — which makes it sensitive to ghosting and blurring.

**SSIM (Structural Similarity Index)**
SSIM compares patches of the image in terms of luminance (mean brightness), contrast (standard deviation), and structure (normalized correlation). It ranges from 0 to 1; values above 0.9 are generally considered good. SSIM is less sensitive to pixel-perfect alignment than PSNR and correlates better with human perceptual judgments in many benchmarks — but it is still a proxy, not a ground truth for perceived quality.

**What low metrics here actually mean.** Poor PSNR or SSIM on training views can indicate:
- The model has not converged (train for more epochs or reduce learning rate).
- The COLMAP initialization was too sparse to seed Gaussians in the right places.
- The image regions contain tissue that moved between frames (violates the static scene assumption of 3DGS).
- The camera calibration has residual error that is larger than the per-Gaussian position precision.


In [ ]:
source_sample_ids = np.linspace(0, len(image_paths) - 1, num=min(3, len(image_paths)), dtype=int)
requested_sample_names = [image_paths[idx].name for idx in source_sample_ids]
frame_name_to_id = {frame.image_path.name: frame_id for frame_id, frame in enumerate(frames)}
sample_ids = [frame_name_to_id[name] for name in requested_sample_names if name in frame_name_to_id]
if not sample_ids:
    print("Requested source samples were not registered by COLMAP; using evenly spaced registered frames.")
    sample_ids = np.linspace(0, len(frames) - 1, num=min(3, len(frames)), dtype=int).tolist()
else:
    print("Rendering source samples:", requested_sample_names)
sample_manifest = workspace / "render_sample_names.txt"
sample_manifest.write_text("\n".join(frames[frame_id].image_path.name for frame_id in sample_ids) + "\n")
metric_rows = []

for frame_id in sample_ids:
    rendered = render_posed_image(model, frames[frame_id], image_scale=0.25, device=device)
    metrics = image_metrics(rendered["render"], rendered["target"])
    metric_rows.append({"frame": frames[frame_id].image_path.name, **metrics})
    show_render_comparison(
        rendered["target"],
        rendered["render"],
        rendered["alpha"],
        metrics=metrics,
    )

display(pd.DataFrame(metric_rows))

## Render An Interpolated COLMAP Trajectory GIF

Rendering the exact training views tells us about reconstruction quality, but the signature capability of 3DGS is **novel-view synthesis** — rendering the scene from camera positions that were never in the training set.

**How camera path interpolation works.**
The anchor frames (taken from the training set) define keyframe poses. Between any two keyframes, we interpolate:
- **Translation**: linear interpolation between camera centres gives a smooth dolly path.
- **Rotation**: spherical linear interpolation (SLERP) on the rotation quaternion gives a smooth, constant-speed rotation — linear interpolation on rotation matrices does not produce constant angular velocity and can distort intermediate poses.

The interpolated cameras are *novel* in the sense that they were never seen during training. If the Gaussians have been placed and shaped correctly, they should still rasterize to a plausible image from these intermediate poses. If the model has overfit to the training views or has holes in coverage, the novel views will show artifacts: floaters (stray Gaussians), holes (regions with no Gaussian coverage), or blurring (Gaussians that are too large and smear across the novel viewpoint).

**Connection to the NeRF/3DGS research community.** This interpolated-path evaluation is the standard way that papers like the original [3DGS paper (Kerbl et al. 2023)](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/) and the Nerfstudio framework (`ns-render interpolate`) demonstrate results. The GIF you produce here is the same kind of artifact, just at smaller scale.


In [ ]:
sample_manifest = workspace / "render_sample_names.txt"
if sample_manifest.exists():
    anchor_names = sample_manifest.read_text().splitlines()
else:
    source_sample_ids = np.linspace(0, len(image_paths) - 1, num=min(3, len(image_paths)), dtype=int)
    anchor_names = [image_paths[idx].name for idx in source_sample_ids]

frame_name_to_id = {frame.image_path.name: frame_id for frame_id, frame in enumerate(frames)}
anchor_frames = [frames[frame_name_to_id[name]] for name in anchor_names if name in frame_name_to_id]
if len(anchor_frames) < 2:
    raise ValueError("Need at least two registered COLMAP anchor frames for an interpolated trajectory.")

novel_cameras = interpolate_camera_path(anchor_frames, steps_per_segment=20, include_keyframes=False)
gif_renders = [
    render_virtual_camera(
        model,
        camera["camtoworld"],
        camera["K"],
        reference_frame=anchor_frames[0],
        image_scale=1.0,
        device=device,
    )["render"]
    for camera in novel_cameras
]
gif_path = save_render_gif(workspace / "colmap_3dgs_trajectory.gif", gif_renders, fps=8)
print("anchor frames:", [frame.image_path.name for frame in anchor_frames])
print("novel views:", len(gif_renders))
DisplayImage(filename=str(gif_path))

## Discussion

Work through these questions after inspecting the outputs above. They are designed to push beyond surface-level observation toward mechanistic understanding.

**1. Which regions become sparse points, and which regions stay empty?**
Look at the 3D scatter plot for the all-frames reconstruction. Do the points cluster on the tissue surface, on tool edges, or near specular highlights? Where are the gaps? Relate the gap locations back to the visual appearance of the images: are those regions smooth, uniformly lit, or near the image boundary?

**2. What changed when using every 5th and every 10th frame?**
Compare the `registered_images`, `sparse_points`, and `mean_reprojection_error` columns in the summary table. Did registration rate drop smoothly, or was there a threshold effect? Did reprojection error go up or down with sparser frames — and what does that tell you about the triangulation angle vs. noise trade-off?

**3. Does the 3DGS render fail where COLMAP has little geometry?**
Overlay your memory of the sparse point cloud with the rendered images. Are the poorly-rendered regions (high per-pixel error, blurry, or missing) the same regions that had few COLMAP points? If yes, this confirms the initialization bottleneck hypothesis. If no, something else is limiting render quality — possibly the number of training epochs, the learning rate schedule, or dynamic tissue.

**4. What changes in the 3DGS reconstruction when using every_5th or all frames?**
*(You need to train a second 3DGS model from the sparser initialization and compare PSNR/SSIM.)* Think through what you predict before doing the experiment: fewer seed points → fewer initial Gaussians → less scene coverage → lower PSNR? Or does the optimizer recover by splitting Gaussians into the empty regions?

**5. Which assumptions of feature-based SfM are weak in endoscopy?**
Classic SfM assumes: a static rigid scene, consistent lighting, texturally rich surfaces, and a calibrated camera with negligible lens distortion. Rank these assumptions from strongest to weakest violation in a laparoscopic procedure, and suggest one concrete technical approach that addresses the worst violation (hint: notebook 03 addresses the scale ambiguity problem using stereo; what would address the textureless surface problem?).
